### 1. 고장 데이터 전처리

In [2]:
# 라이브러리 실행
import pandas as pd
from glob import glob
import os

In [3]:
# 샘플 데이터 컬럼 확인
df = pd.read_csv('raw_data/repair/2102_2106.csv', encoding='cp949')
print(df.columns)

Index(['자전거번호', '등록일시', '고장구분'], dtype='object')


In [4]:
# 전체 파일 목록 확인
repair_files = sorted(glob('raw_data/repair/*.csv'))
repair_files

['raw_data/repair/2102_2106.csv',
 'raw_data/repair/2107_2112.csv',
 'raw_data/repair/2201_2206.csv',
 'raw_data/repair/2207_2212.csv',
 'raw_data/repair/2301_2306.csv',
 'raw_data/repair/2307_2310.csv',
 'raw_data/repair/2311_2312.csv',
 'raw_data/repair/2401_2406.csv',
 'raw_data/repair/2407_2412.csv',
 'raw_data/repair/2501_2506.csv',
 'raw_data/repair/2507_2512.csv']

In [5]:
# 각 데이터 파일별로 컬럼명 확인
# 인코딩 언어 확인
for file in repair_files:
    try:
        df_temp = pd.read_csv(file, encoding='cp949', nrows=3)
    except:
        try:
            df_temp = pd.read_csv(file, encoding='utf-8', nrows=3)
        except:
            df_temp = pd.read_excel(file, nrows=3)

    df_temp.columns = df_temp.columns.str.strip()
    print(f"\n파일명: {file}")
    print(df_temp.columns.tolist())


파일명: raw_data/repair/2102_2106.csv
['자전거번호', '등록일시', '고장구분']

파일명: raw_data/repair/2107_2112.csv
['자전거번호', '등록일시', '고장구분']

파일명: raw_data/repair/2201_2206.csv
['자전거번호', '등록일시', '고장구분']

파일명: raw_data/repair/2207_2212.csv
['자전거번호', '등록일시', '고장구분']

파일명: raw_data/repair/2301_2306.csv
['자전거번호', '등록일시', '구분']

파일명: raw_data/repair/2307_2310.csv
['자전거번호', '등록일시', '고장구분']

파일명: raw_data/repair/2311_2312.csv
['자전거번호', '등록일시', '고장구분']

파일명: raw_data/repair/2401_2406.csv
['자전거번호', '등록일시', '구분']

파일명: raw_data/repair/2407_2412.csv
['자전거번호', '등록일시', '구분']

파일명: raw_data/repair/2501_2506.csv
['자전거번호', '등록일시', '구분']

파일명: raw_data/repair/2507_2512.csv
['자전거번호', '등록일시', '구분']


In [27]:
# 파일 읽기
def read_repair_file(file):
    try:
        return pd.read_csv(file, encoding='cp949')
    except:
        try:
            return pd.read_csv(file, encoding='utf-8')
        except:
            return pd.read_excel(file)

In [28]:
# 고장구분, 구분 컬럼 통일 및 전처리
def preprocess_repair_file(file):
    df = read_repair_file(file)
    df.columns = df.columns.str.strip()

    if '고장구분' in df.columns:
        df = df[['자전거번호', '등록일시', '고장구분']].copy()
        df = df.rename(columns={'고장구분': '구분'})
    elif '구분' in df.columns:
        df = df[['자전거번호', '등록일시', '구분']].copy()
    else:
        print(f'컬럼 확인 필요: {file}')
        print(df.columns.tolist())
        return None

    df['자전거번호'] = df['자전거번호'].astype(str).str.strip()
    df['등록일시'] = pd.to_datetime(df['등록일시'], errors='coerce')
    df['구분'] = df['구분'].astype(str).str.strip()

    df = df.dropna(subset=['자전거번호', '등록일시', '구분']) # 결측치 제거
    df = df.drop_duplicates() # 중복 제거
    df['source_file'] = file.split('/')[-1]

    return df

In [29]:
# 전체 파일에 대하여 전처리 수행
df_repair_list = []

for file in repair_files:
    df_temp = preprocess_repair_file(file)
    if df_temp is not None:
        df_repair_list.append(df_temp)

In [30]:
# 파일 병합
df_repair = pd.concat(df_repair_list, ignore_index=True)
df_repair.head()

,자전거번호,등록일시,구분,source_file
0,SPB-51735,2021-02-01,타이어,2102_2106.csv
1,SPB-52819,2021-02-01,기타,2102_2106.csv
2,SPB-53829,2021-02-01,기타,2102_2106.csv
3,SPB-51114,2021-02-01,단말기,2102_2106.csv
4,SPB-43267,2021-02-01,안장,2102_2106.csv


In [13]:
df_repair.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 777644 entries, 0 to 777643
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   자전거번호        777644 non-null  object        
 1   등록일시         777644 non-null  datetime64[ns]
 2   구분           777644 non-null  object        
 3   source_file  777644 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 23.7+ MB


In [14]:
df_repair.isnull().sum()

자전거번호          0
등록일시           0
구분             0
source_file    0
dtype: int64

In [15]:
df_repair['구분'].value_counts()

구분
기타     225687
체인     155918
타이어    145616
안장     133674
페달      80516
단말기     36233
Name: count, dtype: int64

In [16]:
df_repair['등록일시'].min(), df_repair['등록일시'].max()

(Timestamp('2021-02-01 00:00:00'), Timestamp('2025-12-31 23:43:29'))

In [ ]:
# 파일별 데이터 개수 확인
df_repair['source_file'].value_counts().sort_index()

source_file
2102_2106.csv     53843
2107_2112.csv     95689
2201_2206.csv     74924
2207_2212.csv    105550
2301_2306.csv     83734
2307_2310.csv     67328
2311_2312.csv     21851
2401_2406.csv     76464
2407_2412.csv     85382
2501_2506.csv     53574
2507_2512.csv     59305
Name: count, dtype: int64